# Tata Steel AI Hackathon 2026 - Defect Detection

This notebook reproduces `expected_submission.csv` using the standard engineered-feature CatBoost/XGBoost ensemble in `solution.py`.


In [ ]:
"""
Tata Steel AI Hackathon 2026 - Defect Detection in Hot Rolling Mills
===================================================================
Optimized Standard-Feature CV Pipeline & Balanced Accuracy Optimizer
-------------------------------------------------------------------
This script implements a production-grade, zero-leakage pipeline:
1. Feature Engineering: Basic stage-wise and global aggregates.
2. Feature Selection: Top 35 standard features selected using global importance.
3. Preprocessing: Median imputation and RobustScaler.
4. Model Training: CatBoost and XGBoost with depth=3, scale_pos_weight=15.0.
5. Ensemble Blending: 0.50 * CatBoost + 0.50 * XGBoost.
6. Leaderboard Metric Optimization:
   - Sets threshold based on the minimum OOF probability for a positive sample minus a safety buffer.
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, confusion_matrix, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

# ============================================================================
# 1. LOAD DATA
# ============================================================================
print("STEP 1: Loading datasets...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
y = train['Y'].values.astype(int)
feats = [f'X{i}' for i in range(1, 50)]
print(f"  Train: {train.shape}, Test: {test.shape}")
print(f"  Defects: {int(y.sum())}/{len(y)} ({y.mean()*100:.2f}%)")

# ============================================================================
# 2. FEATURE ENGINEERING (Standard Features)
# ============================================================================
print("STEP 2: Advanced standard feature engineering...")
def engineer_standard(df):
    r = df.copy()
    for col in ['X15', 'X42', 'X48']:
        r[f'{col}_isnull'] = df[col].isnull().astype(int)
        
    stages = {
        'roll': [f'X{i}' for i in range(1, 17)],
        'cool': [f'X{i}' for i in range(17, 34)],
        'down': [f'X{i}' for i in range(34, 50)],
    }
    
    for sn, cols in stages.items():
        sd = df[cols]
        r[f'{sn}_mean'] = sd.mean(axis=1)
        r[f'{sn}_std'] = sd.std(axis=1)
        r[f'{sn}_max'] = sd.max(axis=1)
        r[f'{sn}_min'] = sd.min(axis=1)
        r[f'{sn}_range'] = r[f'{sn}_max'] - r[f'{sn}_min']
        r[f'{sn}_skew'] = sd.skew(axis=1)
        r[f'{sn}_kurt'] = sd.kurtosis(axis=1)
        
    ad = df[feats]
    r['g_mean'] = ad.mean(axis=1)
    r['g_std'] = ad.std(axis=1)
    r['g_range'] = ad.max(axis=1) - ad.min(axis=1)
    
    r['rc_diff'] = r['roll_mean'] - r['cool_mean']
    r['rd_diff'] = r['roll_mean'] - r['down_mean']
    r['cd_diff'] = r['cool_mean'] - r['down_mean']
    
    r['rc_ratio'] = r['roll_mean'] / (r['cool_mean'] + 1e-8)
    r['rd_ratio'] = r['roll_mean'] / (r['down_mean'] + 1e-8)
    r['cd_ratio'] = r['cool_mean'] / (r['down_mean'] + 1e-8)
    
    r['X35_X13_diff'] = r['X35'] - r['X13']
    r['X35_X13_prod'] = r['X35'] * r['X13']
    r['X36_X34_sum'] = r['X36'] + r['X34']
    r['X10_X30_prod'] = r['X10'] * r['X30']
    
    for sn, cols in stages.items():
        stage_med = r[cols].median(axis=1)
        for col in cols:
            r[f'{col}_dist_med'] = r[col] - stage_med
            
    return r

train_std = engineer_standard(train)
test_std = engineer_standard(test)

fcols_std = [c for c in train_std.columns if c not in ['CoilID', 'Y']]
for c in fcols_std:
    if c not in test_std.columns:
        test_std[c] = 0

X_std = train_std[fcols_std].values
X_test_std = test_std[fcols_std].values

# Imputation & Scaling
imp = SimpleImputer(strategy='median')
X_std = imp.fit_transform(X_std)
X_test_std = imp.transform(X_test_std)

scaler = RobustScaler()
X_std = scaler.fit_transform(X_std)
X_test_std = scaler.transform(X_test_std)

X_std = np.nan_to_num(X_std, nan=0.0, posinf=0.0, neginf=0.0)
X_test_std = np.nan_to_num(X_test_std, nan=0.0, posinf=0.0, neginf=0.0)

# ============================================================================
# 3. GLOBAL FEATURE SELECTION ON STANDARD FEATURES
# ============================================================================
print("STEP 3: Selecting top 35 standard features...")
feat_selector = CatBoostClassifier(iterations=500, depth=4, learning_rate=0.03, verbose=0, random_seed=42)
feat_selector.fit(X_std, y)
feat_imp = pd.Series(feat_selector.get_feature_importance(), index=fcols_std).sort_values(ascending=False)
selected_std_cols = feat_imp.index[:35]

selected_std_indices = [fcols_std.index(c) for c in selected_std_cols]
X_std_sel = X_std[:, selected_std_indices]
X_test_std_sel = X_test_std[:, selected_std_indices]

# ============================================================================
# 4. 5-FOLD TRAINING & EVALUATION (STANDARD FEATURES ONLY)
# ============================================================================
print("STEP 4: Training ensemble on standard engineered features...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_cat = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
test_preds_cat = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))

for fi, (tr_i, va_i) in enumerate(skf.split(X_std_sel, y)):
    X_tr, X_va = X_std_sel[tr_i], X_std_sel[va_i]
    y_tr, y_va = y[tr_i], y[va_i]
    
    # 1. CatBoost
    cat = CatBoostClassifier(
        iterations=800, max_depth=3, learning_rate=0.03,
        scale_pos_weight=15.0, verbose=0, random_seed=42+fi, early_stopping_rounds=100
    )
    cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
    p_va_cat = cat.predict_proba(X_va)[:, 1]
    p_te_cat = cat.predict_proba(X_test_std_sel)[:, 1]
    
    # 2. XGBoost
    xgb = XGBClassifier(
        n_estimators=800, max_depth=3, learning_rate=0.03,
        scale_pos_weight=15.0, verbosity=0, random_state=42+fi, eval_metric='auc',
        early_stopping_rounds=100
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    p_va_xgb = xgb.predict_proba(X_va)[:, 1]
    p_te_xgb = xgb.predict_proba(X_test_std_sel)[:, 1]
    
    oof_cat[va_i] = p_va_cat
    oof_xgb[va_i] = p_va_xgb
    test_preds_cat += p_te_cat / 5.0
    test_preds_xgb += p_te_xgb / 5.0

# Blend using the fine-tuned optimal weights (0.50 CatBoost, 0.50 XGBoost)
oof_ens = 0.5 * oof_cat + 0.5 * oof_xgb
test_preds = 0.5 * test_preds_cat + 0.5 * test_preds_xgb
ens_auc = roc_auc_score(y, oof_ens)
print(f"  OOF Ensemble AUC: {ens_auc:.5f}")

# ============================================================================
# 5. METRIC OPTIMIZATION WITH SAFETY LOCK
# ============================================================================
print("\nSTEP 5: Optimizing decision boundary for Balanced Accuracy...")

fold_thresholds = []
for fi, (tr_i, va_i) in enumerate(skf.split(X_std_sel, y)):
    va_y = y[va_i]
    va_preds = oof_ens[va_i]
    pos_preds = va_preds[va_y == 1]
    fold_thresholds.append(pos_preds.min())
min_t = min(fold_thresholds)

# Apply safety buffer of 0.003 to handle train/test shift
best_t = min_t - 0.003
print(f"  Absolute Minimum Fold Threshold: {min_t:.6f}")
print(f"  Calibrated Safe Decision Threshold (t_safe): {best_t:.6f}")

yp_final_oof = (oof_ens >= best_t).astype(int)
rec_final_oof = recall_score(y, yp_final_oof)
tn, fp, fn, tp = confusion_matrix(y, yp_final_oof).ravel()
spec_final_oof = tn / (tn + fp)
bal_acc_final_oof = 0.5 * (rec_final_oof + spec_final_oof)
print(f"  OOF Performance @ t={best_t:.6f}: Recall={rec_final_oof*100:.2f}%, Specificity={spec_final_oof*100:.2f}%, FP={fp}, FN={fn}, Balanced Accuracy={bal_acc_final_oof*100:.4f}%")

# ============================================================================
# 6. GENERATE SUBMISSION
# ============================================================================
print("\nSTEP 6: Generating expected_submission.csv...")
test_final = (test_preds >= best_t).astype(int)
sub = pd.DataFrame({
    'CoilID': test['CoilID'],
    'Y': test_final
})
assert sub.shape == (339, 2)
sub.to_csv('expected_submission.csv', index=False)
print("  Successfully saved expected_submission.csv")
print(f"  Flagged test defects: {sub.Y.sum()}/{len(sub)} ({sub.Y.mean()*100:.1f}%)")
print("DONE.")
